# 04. Path-distance classification

Status: Euclidean and dynamic time warping baselines complete on supervisor-provided BasicMotions data. Signature features remain open.


## 1. Purpose

Classification tests whether a path discrepancy preserves class-relevant shape without involving a trained reconstruction model. Model, optimiser, and training loss are absent, so performance changes can be attributed to representation and distance. Official train and test splits remain fixed.


## 2. Data and normalisation

BasicMotions contains 40 training and 40 test paths. Each path $x^{(i)}\in\mathbb R^{6\times100}$ contains three accelerometer and three gyroscope channels sampled at a uniform cadence. Labels are standing, running, walking, and badminton.

For channel $c$, calculate training statistics

$$
\mu_c=\frac{1}{n_{\mathrm{train}}T}\sum_{i=1}^{n_{\mathrm{train}}}\sum_{r=1}^{T}x^{(i)}_{c,r},\qquad
\sigma_c^2=\frac{1}{n_{\mathrm{train}}T}\sum_{i=1}^{n_{\mathrm{train}}}\sum_{r=1}^{T}(x^{(i)}_{c,r}-\mu_c)^2.
$$

Both splits use $\widetilde x^{(i)}_{c,r}=(x^{(i)}_{c,r}-\mu_c)/\sigma_c$. Test data do not affect normalisation.


## 3. Fixed 1-nearest-neighbour algorithm

For test path $z$, predict label of training path

$$
i^*(z)=\operatorname*{arg\,min}_{1\leq i\leq n_{\mathrm{train}}}d(z,x^{(i)}).
$$

Euclidean distance flattens channel and time coordinates:

$$
d_{\mathrm E}(x,z)=\left(\sum_{c=1}^{6}\sum_{r=1}^{100}|x_{c,r}-z_{c,r}|^2\right)^{1/2}.
$$

Dynamic time warping uses local multichannel cost $\delta(r,s)=\|x_{:,r}-z_{:,s}\|_2^2$ and recursion

$$
D_{r,s}=\delta(r,s)+\min\{D_{r-1,s},D_{r,s-1},D_{r-1,s-1}\}.
$$

Accuracy is fraction of correct test labels. Balanced accuracy is mean class recall.


In [ ]:
import json
from pathlib import Path

result = json.loads(
    Path('../results/runs/classification_basicmotions/results.json').read_text()
)
[(row['distance'], row['scores']) for row in result['results']]


## 4. Results

| distance | accuracy | balanced accuracy |
|---|---:|---:|
| Euclidean | 0.575 | 0.575 |
| dynamic time warping | 0.900 | 0.900 |

DTW gains 32.5 percentage points on this split. BasicMotions classes permit local timing variation, so alignment changes are useful for this dataset. One dataset supports pipeline validity and supplies a control for later representations; it does not establish general superiority of DTW.


## 5. Signature extension

Next representation is a truncated signature or log signature of an explicitly augmented path. Fix augmentation, interpolation, truncation level, and feature scaling using training information. Apply the same 1-nearest-neighbour rule so representation is the only changed component.

Raw signature uses increments and omits absolute level. Initial study therefore includes a base point or initial value. Time augmentation records cadence; lead-lag augmentation is a later alternative when interaction with increments is the intended feature.


## 6. Reproducibility map

- Data: `data/raw/BasicMotions/`
- Configuration: `configs/classification_basicmotions.yaml`
- Classification functions: `src/pathloss/classification.py`
- Runner: `scripts/run_classification.py`
- Stored output: `results/runs/classification_basicmotions/results.json`
